# Fair Compensation Project

### Cleaning and Preprocessing Employment data

The cleaning steps for employment data included: filtering for state level, filtering for detailed occupations (that will be easier to match up with the job and wages dataset occupations), annualize wages for easier comparison against the job and wages dataset (which contains annual wages), and handled the suppression codes in the dataset.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/employment.csv', encoding='latin1')
df.head()


df = df.drop(columns=['PCT_TOTAL', 'PCT_RPT'])
df = df[df['AREA_TYPE'] == 2]
print(f"Rows after filtering for states: {len(df)}")

df = df[df['O_GROUP'] == 'detailed']
print(f"Rows after filtering for detailed occupations: {len(df)}")

# handle BLS suppression codes 
# * for unreliable data, # for wage >= 100/h -> will replace both with NaN
wage_cols = ['H_MEAN', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90', 'A_PCT90']
for col in wage_cols:
    df[col] = df[col].replace({'*': np.nan, '#': np.nan})
    df[col] = pd.to_numeric(df[col], errors = 'coerce')

# annualize hourly wages
hourly_to_annual = {
    'H_MEAN' : 'A_MEAN_CALC',
    'H_MEDIAN' : 'A_MEDIAN_CALC',
    'H_PCT10' : 'A_PCT10_CALC',
    'H_PCT90' : 'A_PCT90_CALC',
}

for h_col, a_col in hourly_to_annual.items():
    df[a_col] = df[h_col] * 2080

print(df[['OCC_TITLE', 'A_MEAN', 'A_MEAN_CALC']].head(10))

print(df[['A_PCT10', 'A_PCT90', 'A_PCT90_CALC']].isnull().sum())

cols_to_keep = {
    'AREA_TITLE': 'state',
    'PRIM_STATE': 'state_abbr',
    'OCC_CODE': 'occ_code',
    'OCC_TITLE': 'occ_title',
    'TOT_EMP': 'total_employed',
    'A_MEAN': 'annual_mean_wage',
    'A_MEDIAN': 'annual_median_wage',
    'A_PCT10': 'annual_p10_wage',
    'A_PCT90_CALC': 'annual_p90_wage',
}
df_clean = df[list(cols_to_keep.keys())].rename(columns=cols_to_keep)
print(df_clean.shape)
df_clean.head()

wage_cols = ['annual_mean_wage', 'annual_median_wage', 'annual_p10_wage', 'annual_p90_wage']
for col in wage_cols:
    df_clean[col] = df_clean[col].astype(str).str.replace(',', '').str.strip()
    df_clean[col] = pd.to_numeric(df_clean[col], errors = 'coerce')

df_clean.to_csv('data/cleaned/employment.csv', index = False)
print("Saved to data/cleaned/employment.csv")
print(df_clean.shape)
df_clean.describe()


Rows after filtering for states: 36643
Rows after filtering for detailed occupations: 35470
                              OCC_TITLE   A_MEAN  A_MEAN_CALC
2                      Chief Executives  221,030     221020.8
3       General and Operations Managers  129,310     129313.6
4                           Legislators   33,690          NaN
5   Advertising and Promotions Managers  112,290     112299.2
6                    Marketing Managers  130,920     130915.2
7                        Sales Managers  137,700     137696.0
8             Public Relations Managers  105,200     105206.4
9                  Fundraising Managers   84,380      84385.6
10     Administrative Services Managers  128,220     128211.2
11                  Facilities Managers  118,510     118518.4
A_PCT10             0
A_PCT90         35470
A_PCT90_CALC     4235
dtype: int64
(35470, 9)
Saved to data/cleaned/employment.csv
(35470, 9)


,annual_mean_wage,annual_median_wage,annual_p10_wage,annual_p90_wage
count,34781.000000,34361.000000,34781.000000,31235.000000
mean,70092.097410,62886.243707,43830.104367,87819.466573
std,43891.345124,30402.783097,20986.446552,41551.454917
min,19630.000000,16280.000000,15080.000000,21278.400000
25%,44260.000000,41500.000000,31060.000000,57532.800000
50%,58450.000000,54740.000000,38220.000000,76876.800000
75%,82010.000000,76880.000000,50050.000000,107088.800000
max,581560.000000,239060.000000,238630.000000,239054.400000


### Cleaning and Preprocessing Personal Consumption Expenditures data

In [2]:
df = pd.read_csv('data/raw/pce.csv')
print(df.shape)
print(df['Description'].unique())
df.head()

df = df.dropna(subset = ['GeoName'])
df = df[df['LineCode'] == 1]
print(f"Rows after filtering for total PCE: {len(df)}")
print(df['Description'].unique())

us_states = [
    'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia', 'Hawaii', 
    'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi',
    'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma',
    'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virgina', 'Washington', 'West Virginia', 
    'Wisconsin', 'Wyoming'
]

df = df[df['GeoName'].isin(us_states)]
print(f"Rows after filtering for states: {len(df)}")

cols_to_keep = {
    'GeoName': 'state',
    '2023': 'pce_2023_in_millions'
}

df_clean = df[list(cols_to_keep.keys())].rename(columns = cols_to_keep)
df_clean = df_clean.reset_index(drop = True)

print(df_clean.shape)
print(df_clean.isnull().sum())
df_clean.head(10)

# sanity check
print(df_clean.sort_values('pce_2023_in_millions', ascending = False)) # will need to divide by state population to get per capita numbers

df_clean.to_csv('data/cleaned/pce.csv', index = False)
print("Saved to data/cleaned/pce.csv")

    

(1444, 35)
<StringArray>
[                                                     'Personal consumption expenditures ',
                                                                                 ' Goods ',
                                                                        '  Durable goods ',
                                                            '   Motor vehicles and parts ',
                                         '   Furnishings and durable household equipment ',
                                                     '   Recreational goods and vehicles ',
                                                                 '   Other durable goods ',
                                                                     '  Nondurable goods ',
                           '   Food and beverages purchased for off-premises consumption ',
                                                               '   Clothing and footwear ',
                                                     ' 

### Cleaning and Preprocessing Jobs and Wages data

The cleaning steps for jobs and wages data (companies.csv) included: removing 14 duplicate companies ('11:59:00'), encoded revenue and employee size bands as integers, 'Director' 'Director_Score 'URL' columns dropped due to having too many null values and being less relevant, stripped and title-cased 'Company' for easy joining with jobs.csv data

In [3]:
df = pd.read_csv('data/raw/companies.csv', encoding = 'utf-16', on_bad_lines = 'skip')
print(df.shape)
df.head()

size_order = ['XXXS', 'XXS', 'XS', 'S', 'M', 'L', 'XL', 'XXL', 'XXXL']
size_map = {v: i for i, v in enumerate(size_order)}

df['revenue_rank'] = df['Revenue'].map(size_map)
df['employee_rank'] = df['Employee'].map(size_map)

print(df[['Revenue', 'revenue_rank', 'Employee', 'employee_rank']].drop_duplicates().sort_values('revenue_rank'))

print(df.isnull().sum())

df = df.drop(columns = ['Director', 'Director_Score', 'URL'])
print(f"\nShape after dropping sparse columns: {df.shape}")

df = df.rename(columns = {
    'Company': 'company',
    'Sector': 'sector',
    'Sector_Group': 'sector_group', 
    'Revenue': 'revenue_band',
    'Employee': 'employee_band',
    'Company_Score': 'company_score',
    'Reviews': 'reviews_count',
})

df['company_clean'] = df['company'].str.strip().str.title()
print(df[['company', 'company_clean']].head(10))

df_clean = df.groupby('company_clean').agg({
    'company': 'first',
    'sector': 'first',
    'sector_group': 'first',
    'revenue_band': 'first',
    'revenue_rank': 'first',
    'employee_band': 'first',
    'employee_rank': 'first',
    'company_score': 'mean',
    'reviews_count': 'max',
}).reset_index()

df_clean.to_csv('data/cleaned/companies.csv', index = False)
print("Saved to data/cleaned/companies.csv")
print(df_clean.shape)
df_clean.head()




(32623, 10)
     Revenue  revenue_rank Employee  employee_rank
125     XXXS           0.0     XXXL            8.0
1261    XXXS           0.0        M            4.0
1209    XXXS           0.0        L            5.0
1117    XXXS           0.0      XXL            7.0
853     XXXS           0.0       XL            6.0
...      ...           ...      ...            ...
4806     NaN           NaN     XXXL            8.0
4827     NaN           NaN        L            5.0
4835     NaN           NaN        M            4.0
4880     NaN           NaN     XXXS            0.0
8328     NaN           NaN      XXS            1.0

[89 rows x 4 columns]
Company               0
Sector            11001
Sector_Group      11001
Revenue           21071
Employee          15850
Company_Score     10471
Reviews           10471
Director          25605
Director_Score    27074
URL               19219
revenue_rank      21071
employee_rank     15850
dtype: int64

Shape after dropping sparse columns: (32623, 9)
   

,company_clean,company,sector,sector_group,revenue_band,revenue_rank,employee_band,employee_rank,company_score,reviews_count
0,"""K"" Line America, Inc.","""K"" Line America, Inc.",NaN,NaN,NaN,NaN,NaN,NaN,5.0,1.0
1,#Teamgohealth,#TeamGoHealth,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,#Twiceasnice Recruiting,#twiceasnice Recruiting,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,(Qsi) - Quality Systems Integrators,(QSI) - Quality Systems Integrators,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"(Saipsit, Inc. Has Multiple Openings In Housto...","(Saipsit, Inc. has multiple openings in Housto...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The cleaning steps for job and wages data (jobs.csv) included: dropping 22 rows where annual mean < 15000, dropped rows with null for 'State', dropped non US states (puerto rico, armed forces, northern mariana islands, and us minor outlying islands), filled null in 'Profile' with 'Unspecified', for null 'Remote' values I filled with 'On-site' since that is usually the default, created skills list and skills count out of the 'Skills' column, mapped Jobs_Group to the relevant BLS occupation codes used in employment.csv, added 'company_clean' to strip whitespace and title-casing from 'Company' which can then be used for joining data

In [4]:
import ast

df = pd.read_csv('data/raw/jobs.csv', encoding = 'utf-16', on_bad_lines = 'skip')
print(df.shape)
df.head()

df['annual_low'] = df['Low_Salary']
df['annual_mean'] = df['Mean_Salary']
df['annual_high'] = df['High_Salary']

print("annual_mean stats:")
print(df['annual_mean'].describe())
print()

mask = df['annual_mean'].notna()
print("Below 15k:", (df.loc[mask, 'annual_mean'] < 15000).sum())
print("Above 1M:", (df.loc[mask, 'annual_mean'] > 1000000).sum())

df = df[df['annual_mean'].isna()| ((df['annual_mean'] >= 15000) & (df['annual_mean'] <= 1000000))]
print(f"Shape after salary filter: {df.shape}")

valid_states = [
    'AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 
    'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA',
    'VT', 'WA', 'WI', 'WV', 'WY']

df = df[df['State'].isin(valid_states)]
print(f"Shape after state filter: {df.shape}")
print(f"Unique states: {df['State'].nunique()}")

df['Profile'] = df['Profile'].fillna('Unspecified')
df['Remote'] = df['Remote'].fillna('On-site')

def parse_skills(s):
    try:
        result = ast.literal_eval(s)
        if isinstance(result, list):
            return result
        return []
    except (ValueError, SyntaxError):
        return []

df['skills_list'] = df['Skills'].apply(parse_skills)
df['skills_count'] = df['skills_list'].apply(len)

from collections import Counter
all_skills = [s for sublist in df['skills_list'] for s in sublist]
print("Top 15 skills:")
print(Counter(all_skills).most_common(15))

jobs_to_bls = {
    'Data Scientist': '15-2051',
    'Data Analyst': '15-2051',
    'Data Engineer': '15-1243',
    'ML/AL Engineer': '15-1252',
    'Business Analyst': '13-1111',
    'Business Intelligence': '15-2031',
    'Financial Analyst': '13-2051',
    'Operations Analyst': '15-2031',
    'CFO': '11-3031',
    'Controller': '11-3031',
    'Finance': '13-2051',
    'Statistician/Mathematics': '15-2041',
    'Analyst': '13-1111',
    'Others': None,
}

df['bls_occ_code']= df['Jobs_Group'].map(jobs_to_bls)
print("Rows with no BLS mapping:", df['bls_occ_code'].isna().sum())

df['company_clean'] = df['Company'].str.strip().str.title()    

cols = {
    'ID': 'job_id',
    'Job': 'job_title',
    'Jobs_Group': 'jobs_group',
    'Profile': 'seniority',
    'Remote': 'remote',
    'company_clean': 'company_clean',
    'Company': 'company',
    'City': 'city',
    'State': 'state',
    'annual_low': 'annual_low',
    'annual_mean': 'annual_mean',
    'annual_high': 'annual_high',
    'bls_occ_code': 'bls_occ_code',
    'skills_list': 'skills',
    'skills_count': 'skills_count',
}

df_clean = df.rename(columns = cols)[list(cols.values())]

df_clean.to_csv('data/cleaned/jobs.csv', index = False)
print("Saved to data/cleaned/jobs.csv")
print(df_clean.shape)
df_clean.head()




(107001, 15)
annual_mean stats:
count     45078.000000
mean     104926.678051
std       43870.769747
min        9200.000000
25%       75000.000000
50%       96600.000000
75%      125000.000000
max      600000.000000
Name: annual_mean, dtype: float64

Below 15k: 22
Above 1M: 0
Shape after salary filter: (106979, 18)
Shape after state filter: (97352, 18)
Unique states: 51
Top 15 skills:
[('Bachelor', 64305), ('Office', 34019), ('Excel', 28156), ('SQL', 26643), ('Master', 18244), ('Python', 17682), ('Word', 12464), ('Tableau', 12263), ('PowerPoint', 12258), ('CPA', 11952), ('Power BI', 11915), ('ERP', 7644), ('Access', 7522), ('Agile', 7424), ('SAP', 7057)]
Rows with no BLS mapping: 5449
Saved to data/cleaned/jobs.csv
(97352, 15)


,job_id,job_title,jobs_group,seniority,remote,company_clean,company,city,state,annual_low,annual_mean,annual_high,bls_occ_code,skills,skills_count
0,sj_1e37379f40861c74,Business Analyst,Business Analyst,Unspecified,On-site,Cybercoders,CyberCoders,Torrington,CT,80000.0,95000.0,110000.0,13-1111,[],0
1,sj_a2789bdbc24f4aed,RPA Business Systems Analyst,Business Analyst,Unspecified,On-site,Amerihealth,Amerihealth,Philadelphia,PA,NaN,NaN,NaN,13-1111,"[Office, SQL, Bachelor]",3
2,job_15e7be7c9bf658e3,Quantitive Business Analyst - Strategic Data S...,Business Analyst,Unspecified,On-site,Apple,Apple,Austin,TX,NaN,NaN,NaN,13-1111,"[Python, SQL, Bachelor]",3
3,job_e8519e1ec2d60a16,Business Line Product Lifecycle Management (PL...,Business Analyst,Junior,On-site,Nxp Semiconductors,NXP Semiconductors,Austin,TX,NaN,NaN,NaN,13-1111,[Bachelor],1
4,job_0545baf6560877d1,Global Markets Operations Asset Services Ops S...,Operations Analyst,Senior,On-site,Bank Of America,Bank of America,Jacksonville,FL,NaN,NaN,NaN,15-2031,[Excel],1
